<a href="https://colab.research.google.com/github/cras-lab/LangChain/blob/main/NewsBriefingAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

실습: LLM이 RSS를 직접 고르는 뉴스 브리핑 Agent<BR>
필요한 모듈 설치



In [39]:
pip install -qU feedparser pandas langchain langchain-core langchain-openai

팔요한 모듈 모두 임포트

In [40]:
import re     # Regular Expression
import html
import json
import getpass
import feedparser # 뉴스피드를 Python 데이터 구조로 변환
import pandas as pd

from IPython.display import display # DF를 보기좋게 출력
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

외부에서 키를 입력받는다.

In [ ]:
openai_api_key = getpass.getpass("OpenAI API Key 입력: ")

RSS 카탈로그들

In [42]:
RSS_CATALOG = {
    # -------------------------
    # 한국어 뉴스 - SBS
    # -------------------------
    "SBS 최신": {
        "url": "https://news.sbs.co.kr/news/newsflashRssFeed.do?plink=RSSREADER",
        "language": "ko",
        "region": "Korea",
        "description": "한국 최신 종합 뉴스. 정치, 경제, 사회, 국제 이슈가 섞여 있음."
    },
    "SBS 이슈": {
        "url": "https://news.sbs.co.kr/news/headlineRssFeed.do?plink=RSSREADER",
        "language": "ko",
        "region": "Korea",
        "description": "한국 주요 헤드라인과 이슈 중심 뉴스."
    },
    "SBS 정치": {
        "url": "https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=01&plink=RSSREADER",
        "language": "ko",
        "region": "Korea",
        "description": "한국 정치, 국회, 정부, 대통령, 선거 관련 뉴스."
    },
    "SBS 경제": {
        "url": "https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=02&plink=RSSREADER",
        "language": "ko",
        "region": "Korea",
        "description": "한국 경제, 금융, 기업, 산업, 부동산, 소비 관련 뉴스."
    },
    "SBS 사회": {
        "url": "https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=03&plink=RSSREADER",
        "language": "ko",
        "region": "Korea",
        "description": "한국 사회, 사건, 교육, 노동, 생활 관련 뉴스."
    },
    "SBS 국제": {
        "url": "https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=07&plink=RSSREADER",
        "language": "ko",
        "region": "Global",
        "description": "국제정세, 미국, 중국, 전쟁, 외교, 글로벌 이슈 뉴스."
    },
    "SBS 생활문화": {
        "url": "https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=08&plink=RSSREADER",
        "language": "ko",
        "region": "Korea",
        "description": "생활, 문화, 건강, 과학, 교육, 사회 트렌드 관련 뉴스."
    },

    # -------------------------
    # 한국 관련 영어 뉴스 - KBS World
    # -------------------------
    "KBS World Today": {
        "url": "http://world.kbs.co.kr/rss/rss_news.htm?lang=e",
        "language": "en",
        "region": "Korea",
        "description": "한국 관련 주요 뉴스를 영어로 제공. 한국 이슈를 영어 기사로 볼 때 적합."
    },
    "KBS World Politics": {
        "url": "http://world.kbs.co.kr/rss/rss_news.htm?lang=e&id=Po",
        "language": "en",
        "region": "Korea",
        "description": "한국 정치와 외교 관련 영어 뉴스."
    },
    "KBS World Economy": {
        "url": "http://world.kbs.co.kr/rss/rss_news.htm?lang=e&id=Ec",
        "language": "en",
        "region": "Korea",
        "description": "한국 경제, 수출, 산업, 금융 관련 영어 뉴스."
    },
    "KBS World International": {
        "url": "http://world.kbs.co.kr/rss/rss_news.htm?lang=e&id=In",
        "language": "en",
        "region": "Global",
        "description": "국제 이슈와 한국 외교 관련 영어 뉴스."
    },
    "KBS World Science": {
        "url": "http://world.kbs.co.kr/rss/rss_news.htm?lang=e&id=Sc",
        "language": "en",
        "region": "Korea",
        "description": "과학, 기술, 보건, 연구 관련 영어 뉴스."
    },

    # -------------------------
    # 글로벌 영어 뉴스 - BBC
    # -------------------------
    "BBC Top": {
        "url": "https://feeds.bbci.co.uk/news/rss.xml",
        "language": "en",
        "region": "Global",
        "description": "글로벌 주요 헤드라인. 국제정세, 경제, 사회, 기술 이슈가 섞여 있음."
    },
    "BBC World": {
        "url": "https://feeds.bbci.co.uk/news/world/rss.xml",
        "language": "en",
        "region": "Global",
        "description": "세계 뉴스. 미국, 중국, 유럽, 전쟁, 외교, 지정학 이슈에 적합."
    },
    "BBC Business": {
        "url": "https://feeds.bbci.co.uk/news/business/rss.xml",
        "language": "en",
        "region": "Global",
        "description": "글로벌 비즈니스, 금융, 금리, 기업, 시장, 거시경제 뉴스."
    },
    "BBC Technology": {
        "url": "https://feeds.bbci.co.uk/news/technology/rss.xml",
        "language": "en",
        "region": "Global",
        "description": "AI, 빅테크, 플랫폼, 반도체, 디지털 기술 관련 뉴스."
    },
    "BBC Science": {
        "url": "https://feeds.bbci.co.uk/news/science_and_environment/rss.xml",
        "language": "en",
        "region": "Global",
        "description": "과학, 기후, 환경, 에너지, 연구 관련 뉴스."
    },

    # -------------------------
    # 글로벌 영어 뉴스 - Guardian
    # -------------------------
    "Guardian World": {
        "url": "https://www.theguardian.com/world/rss",
        "language": "en",
        "region": "Global",
        "description": "국제정치, 전쟁, 외교, 사회 이슈 중심 글로벌 뉴스."
    },
    "Guardian Business": {
        "url": "https://www.theguardian.com/business/rss",
        "language": "en",
        "region": "Global",
        "description": "경제, 금융시장, 기업, 인플레이션, 금리, 노동시장 관련 뉴스."
    },
    "Guardian Technology": {
        "url": "https://www.theguardian.com/technology/rss",
        "language": "en",
        "region": "Global",
        "description": "AI, 빅테크, 인터넷, 플랫폼, 데이터, 기술 규제 관련 뉴스."
    },
    "Guardian Environment": {
        "url": "https://www.theguardian.com/environment/rss",
        "language": "en",
        "region": "Global",
        "description": "기후변화, 에너지, 탄소, 환경정책, 자연재해 관련 뉴스."
    },
    "Guardian Science": {
        "url": "https://www.theguardian.com/science/rss",
        "language": "en",
        "region": "Global",
        "description": "과학, 연구, 보건, 우주, 생명과학 관련 뉴스."
    },

    # -------------------------
    # 글로벌 영어 뉴스 - Le Monde English
    # -------------------------
    "Le Monde International": {
        "url": "https://www.lemonde.fr/en/international/rss_full.xml",
        "language": "en",
        "region": "Global",
        "description": "국제정세, 유럽, 미국, 중국, 외교, 지정학 관련 뉴스."
    },
    "Le Monde Economy": {
        "url": "https://www.lemonde.fr/en/economy/rss_full.xml",
        "language": "en",
        "region": "Global",
        "description": "경제, 산업, 기업, 금융, 유럽 경제 관련 뉴스."
    },
    "Le Monde Pixels": {
        "url": "https://www.lemonde.fr/en/pixels/rss_full.xml",
        "language": "en",
        "region": "Global",
        "description": "디지털 기술, 플랫폼, AI, 인터넷 문화, 기술 규제 관련 뉴스."
    },
    "Le Monde Science": {
        "url": "https://www.lemonde.fr/en/science/rss_full.xml",
        "language": "en",
        "region": "Global",
        "description": "과학, 보건, 연구, 환경, 기후 관련 뉴스."
    },
}


RSS 수집 옵션 설정

In [43]:
MAX_RSS_TO_SELECT = 8     # 선택할 RSS 개수
ARTICLES_PER_FEED = 20    # 하나의 피드당 기사 수 제한
MAX_ARTICLES_FOR_LLM = 70  # LLM이 선택할 최대 기사 수

RSS_CATALOG 딕셔너리에서 언어,지역, 설명 영역추출

In [44]:
rss_catalog_text = ""

for name, meta in RSS_CATALOG.items():
    rss_catalog_text += f"""
[RSS 이름] {name}
[언어] {meta["language"]}
[지역] {meta["region"]}
[설명] {meta["description"]}
---
"""

외부에서 입력받은 키워드에 따라 어떤 RSS를 사용할 것인지 판단하기 위해 LLM에게 요청하기 위해 프롬프트 설정

In [45]:
router_prompt = ChatPromptTemplate.from_template("""
너는 뉴스 수집 라우터다.

학생이 입력한 키워드 리스트를 보고, 아래 RSS 카탈로그 중 어떤 RSS를 수집해야 하는지 선택하라.

선택 기준:
- 키워드와 직접 관련된 RSS를 우선 선택하라.
- 한국어 키워드가 많으면 한국어 RSS와 한국 관련 영어 RSS를 우선 고려하라.
- 글로벌 키워드가 많으면 BBC, Guardian, Le Monde 등 글로벌 RSS를 고려하라.
- 기술 키워드는 Technology, Science 계열 RSS를 우선 고려하라.
- 금융, 금리, 환율, 소비, 부동산 키워드는 Economy, Business 계열 RSS를 우선 고려하라.
- 정치, 선거, 미국, 중국, 전쟁, 외교 키워드는 Politics, World, International 계열 RSS를 우선 고려하라.
- 너무 많이 고르지 말고 최대 {max_rss_to_select}개만 선택하라.
- RSS 이름은 반드시 카탈로그에 있는 이름과 정확히 일치해야 한다.

키워드:
{student_keywords}

RSS 카탈로그:
{rss_catalog_text}

반드시 아래 JSON 형식으로만 답하라.
마크다운 코드블록을 쓰지 마라.
설명 문장을 JSON 밖에 쓰지 마라.

{{
  "selected_feeds": [
    {{
      "name": "RSS 이름",
      "reason": "선택 이유",
      "expected_relevance": "high 또는 medium 또는 low"
    }}
  ],
  "expanded_keywords": ["키워드 확장어1", "키워드 확장어2"],
  "routing_summary": "이 RSS 조합을 선택한 전체 이유"
}}
""")

외부에서 검색 키워드를 입력받음

In [ ]:
raw_keywords = input(
    "\n분석할 키워드 리스트를 쉼표로 입력하세요.\n"
    "예: AI, 반도체, 금리, 환율\n"
    "예: Trump, China, chip, tariff\n"
    "예: 부동산, 가계부채, 은행, 소비\n"
    "입력: "
).strip()

if raw_keywords:
    student_keywords = [
        k.strip()
        for k in re.split(r"[,，、/|]+", raw_keywords)
        if k.strip()
    ]
else:
    student_keywords = ["전체 뉴스"]

print("\n학생 입력 키워드:")
print(student_keywords)

LLM 모델 선택 및 instantiation. 정확성을 위해 nano대신 5버전 사용

In [47]:
llm = ChatOpenAI(
    model="gpt-5",
    api_key=openai_api_key,
    timeout=90,
    temperature=0,
    max_retries=2,
)

프롬프트를 호출해서 응답추출

In [ ]:
router_chain = router_prompt | llm | StrOutputParser()

router_raw = router_chain.invoke({
    "student_keywords": ", ".join(student_keywords),
    "rss_catalog_text": rss_catalog_text,
    "max_rss_to_select": MAX_RSS_TO_SELECT
})

print("\n" + "=" * 100)
print("LLM RSS 라우팅 원문")
print("=" * 100)
print(router_raw)

JSON 파싱 후 JSON 오류일 경우 기본RSS사용

In [49]:
# ------------------------------------------------------------
# 8. RSS 라우팅 결과 JSON 파싱
# ------------------------------------------------------------

router_clean = router_raw.strip()
router_clean = router_clean.replace("```json", "").replace("```", "").strip()

try:
    router_result = json.loads(router_clean)

except Exception as e:  # 오류의 경우 기본사용
    print("\nJSON 파싱 실패. 에러:", e)
    print("LLM 출력이 JSON 형식이 아닙니다. 기본 RSS를 사용합니다.")

    router_result = {
        "selected_feeds": [
            {
                "name": "SBS 최신",
                "reason": "fallback 기본 RSS",
                "expected_relevance": "medium"
            },
            {
                "name": "SBS 경제",
                "reason": "fallback 기본 RSS",
                "expected_relevance": "medium"
            },
            {
                "name": "BBC Top",
                "reason": "fallback 기본 RSS",
                "expected_relevance": "medium"
            },
            {
                "name": "BBC Business",
                "reason": "fallback 기본 RSS",
                "expected_relevance": "medium"
            },
            {
                "name": "BBC Technology",
                "reason": "fallback 기본 RSS",
                "expected_relevance": "medium"
            },
        ],
        "expanded_keywords": student_keywords,
        "routing_summary": "LLM 라우팅 JSON 파싱 실패로 기본 RSS 사용"
    }


오류가 아니면 LLM 지정 RSS사용

In [50]:
selected_feed_rows = []
selected_rss_names = []

for item in router_result.get("selected_feeds", []):
    name = item.get("name", "").strip()

    if name in RSS_CATALOG and name not in selected_rss_names:
        selected_rss_names.append(name)
        selected_feed_rows.append({
            "selected_rss": name,
            "expected_relevance": item.get("expected_relevance", ""),
            "reason": item.get("reason", ""),
            "url": RSS_CATALOG[name]["url"]
        })

# LLM이 유효한 RSS를 선정하지 않았으면 기본 RSS사용
if len(selected_rss_names) == 0:
    print("\nLLM이 유효한 RSS를 선택하지 못했습니다. 기본 RSS를 사용합니다.")

    selected_rss_names = [
        "SBS 최신",
        "SBS 경제",
        "BBC Top",
        "BBC Business",
        "BBC Technology",
    ]

    selected_feed_rows = []

    for name in selected_rss_names:
        selected_feed_rows.append({
            "selected_rss": name,
            "expected_relevance": "medium",
            "reason": "fallback 기본 RSS",
            "url": RSS_CATALOG[name]["url"]
        })

selected_feeds_df = pd.DataFrame(selected_feed_rows)

expanded_keywords = router_result.get("expanded_keywords", [])
if not expanded_keywords:
    expanded_keywords = student_keywords

expanded_keywords = [
    str(k).strip()
    for k in expanded_keywords
    if str(k).strip()
]

routing_summary = router_result.get("routing_summary", "")


LLM이 선택한 RSS 결과를 요약 출력

In [ ]:
print("\n" + "=" * 100)
print("검증된 RSS 선택 결과")
print("=" * 100)
display(selected_feeds_df)

print("\nLLM이 확장한 키워드:")
print(expanded_keywords)

print("\n라우팅 요약:")
print(routing_summary)

선택된 RSS 내에서만 집중적으로 기사수집

In [ ]:
# ------------------------------------------------------------
# 9. 선택된 RSS만 집중 수집
# ------------------------------------------------------------

rows = []

request_headers = {
    "User-Agent": "Mozilla/5.0 RSS-News-Briefing-Classroom/1.0"
}

for source_name in selected_rss_names:
    feed_url = RSS_CATALOG[source_name]["url"]

    try:
        feed = feedparser.parse(feed_url, request_headers=request_headers)

        print(f"{source_name}: {len(feed.entries)}개 수집")

        for entry in feed.entries[:ARTICLES_PER_FEED]:
            title = entry.get("title", "")
            summary = entry.get("summary", "")
            link = entry.get("link", "")
            published = entry.get("published", "")

            title = html.unescape(re.sub("<.*?>", "", str(title))).strip()
            summary = html.unescape(re.sub("<.*?>", "", str(summary))).strip()
            link = str(link).strip()
            published = str(published).strip()

            if title == "" and summary == "":
                continue

            rows.append({
                "source": source_name,
                "title": title,
                "summary": summary,
                "published": published,
                "link": link
            })

    except Exception as e:
        print(f"{source_name} 수집 실패:", e)

news_df = pd.DataFrame(rows)

if len(news_df) == 0:
    raise RuntimeError("선택된 RSS에서 기사를 가져오지 못했습니다. RSS 주소 또는 네트워크를 확인하세요.")

news_df = news_df.drop_duplicates(subset=["title", "link"]).reset_index(drop=True)

news_df["search_text"] = (
    news_df["title"].fillna("") + " " +
    news_df["summary"].fillna("") + " " +
    news_df["source"].fillna("")
).str.lower()

print("\n선택 RSS 기준 수집 기사 수:", len(news_df))
display(news_df[["source", "title", "published"]].head(50))

키워드와 기사의 유사도를 계산해 유사성 높은 것을 참고해 선택

In [ ]:
# ------------------------------------------------------------
# 10. 기사별 관련도 점수 계산
# ------------------------------------------------------------
# 학생 원 키워드 + LLM 확장 키워드 모두 사용
# 단, 점수는 단순한 설명용 heuristic임
# LLM 최종 브리핑에서 이 점수를 절대적 기준으로 쓰지 않도록 지시함

all_scoring_keywords = []

for k in student_keywords + expanded_keywords:
    k = str(k).strip()
    if k and k not in all_scoring_keywords:
        all_scoring_keywords.append(k)

scored_rows = []

for idx, row in news_df.iterrows():
    title_lower = str(row["title"]).lower()
    summary_lower = str(row["summary"]).lower()

    score = 0
    hit_keywords = []

    for kw in all_scoring_keywords:
        kw_lower = kw.lower()

        if kw_lower == "전체 뉴스":
            continue

        if kw_lower in title_lower:
            score += 5
            hit_keywords.append(kw)

        elif kw_lower in summary_lower:
            score += 3
            hit_keywords.append(kw)

    scored_rows.append({
        "idx": idx,
        "relevance_score": score,
        "hit_keywords": ", ".join(sorted(set(hit_keywords)))
    })

score_df = pd.DataFrame(scored_rows)

news_df = news_df.reset_index().rename(columns={"index": "original_idx"})
news_df = news_df.merge(score_df, left_on="original_idx", right_on="idx", how="left")

briefing_df = news_df.sort_values(
    by=["relevance_score"],
    ascending=False
).head(MAX_ARTICLES_FOR_LLM).copy()

briefing_df = briefing_df.reset_index(drop=True)
briefing_df["article_no"] = briefing_df.index + 1

print("\n관련도 상위 기사:")
display(
    briefing_df[
        [
            "article_no",
            "relevance_score",
            "hit_keywords",
            "source",
            "title",
            "published",
            "link"
        ]
    ].head(60)
)


LLM에 최종 브리핑 입력데이터 구성

In [ ]:
# ------------------------------------------------------------
# 11. LLM 최종 브리핑 입력 데이터 만들기
# ------------------------------------------------------------

student_keyword_text = "\n".join([f"- {kw}" for kw in student_keywords])
expanded_keyword_text = "\n".join([f"- {kw}" for kw in expanded_keywords])

selected_rss_text = ""

for _, row in selected_feeds_df.iterrows():
    selected_rss_text += f"""
[RSS] {row["selected_rss"]}
[예상 관련도] {row["expected_relevance"]}
[선택 이유] {row["reason"]}
[URL] {row["url"]}
---
"""

news_text = ""

for _, row in briefing_df.iterrows():
    news_text += f"""
[기사 번호: {row["article_no"]}]
[관련도 점수] {row["relevance_score"]}
[매칭 키워드] {row["hit_keywords"]}
[출처] {row["source"]}
[제목] {row["title"]}
[요약] {row["summary"]}
[발행시각] {row["published"]}
[링크] {row["link"]}
---
"""

print("\nLLM 최종 브리핑 입력 기사 수:", len(briefing_df))
print("\n최종 브리핑 입력 뉴스 텍스트 일부:")
print(news_text[:5000])

최종 브리핑으 ㄹ위한 프롬프트 구성

In [58]:
# ------------------------------------------------------------
# 12. 최종 브리핑 Prompt
# ------------------------------------------------------------

briefing_prompt = ChatPromptTemplate.from_template("""
너는 RSS 뉴스피드 기반 AI 뉴스 브리핑 애널리스트다.

이번 실습 구조:
1. 학생이 키워드 리스트를 입력했다.
2. LLM이 RSS 카탈로그를 보고 적합한 RSS를 직접 선택했다.
3. 선택된 RSS만 집중 수집했다.
4. 기사별 관련도 점수를 계산했다.
5. 아래 기사 데이터를 바탕으로 최종 브리핑을 작성한다.

중요한 제약:
- 아래 데이터는 RSS의 제목, 요약, 링크, 발행시각이다.
- 기사 본문 전체가 아니다.
- 모든 판단은 반드시 "RSS 기준"이라고 명시하라.
- 기사에 없는 사실은 만들지 말라.
- 투자 추천, 매수/매도 추천은 하지 말라.
- 정치적 선호나 가치판단을 하지 말라.
- 중요한 주장에는 반드시 기사 번호를 근거로 표시하라.
- 한국어와 영어 기사가 섞여 있으면 한국어로 통합 요약하라.
- 관련도 점수는 단순 키워드 기반 heuristic이다.
- 점수가 높다고 반드시 중요한 기사는 아니므로, 기사 내용과 학생 키워드의 의미를 함께 고려하라.

[학생 입력 키워드]
{student_keyword_text}

[LLM 확장 키워드]
{expanded_keyword_text}

[LLM이 선택한 RSS]
{selected_rss_text}

[RSS 선택 라우팅 요약]
{routing_summary}

[RSS 기사 데이터]
{news_text}

[출력 형식]

# LLM RSS 선택 기반 뉴스 브리핑

## 1. Executive Summary
- RSS 기준으로 전체 흐름을 5문장 이내로 요약하라.
- 학생 키워드와 연결되는 핵심 흐름을 중심으로 정리하라.

## 2. RSS 선택 결과 평가
- 선택된 RSS:
- 왜 이 RSS들이 선택되었는가:
- 선택이 적절했던 점:
- 선택의 한계:
- 더 추가하면 좋을 RSS 유형:

## 3. 키워드별 브리핑

학생 입력 키워드 각각에 대해 별도 항목으로 작성하라.

각 키워드별 출력 형식:

### 키워드: [키워드명]
- 관련도: 높음 / 중간 / 낮음 / 관련 기사 없음
- 핵심 내용:
- 근거 기사 번호:
- 직접 관련 기사:
- 간접 관련 기사:
- 전체 뉴스 흐름에서의 의미:
- 추가 확인 필요 데이터:

## 4. 오늘의 중요 이슈 Top 5
학생 키워드와의 관련성을 기준으로 선정하라.

각 이슈별 출력 형식:
- 순위:
- 이슈:
- 관련 키워드:
- 근거 기사 번호:
- 왜 중요한가:
- RSS 기준 한계:

## 5. 키워드 간 연결 구조
- 키워드들이 서로 어떻게 연결되는지 설명하라.
- 예: 금리 → 환율 → 반도체 수출 → 기업 실적
- 연결성이 약하면 약하다고 명확히 말하라.
- 근거 기사 번호를 제시하라.

## 6. 시장·산업 관점 요약
- 긍정 요인:
- 부정 요인:
- 불확실성:
- 단기 관전 포인트:
- 중장기 관전 포인트:

## 7. 수업 토론 질문
- 질문 1:
- 질문 2:
- 질문 3:

## 8. 분석 한계
- RSS 제목/요약만 기반으로 했기 때문에 생기는 한계:
- LLM이 RSS를 선택하는 방식의 장점:
- LLM이 RSS를 선택하는 방식의 위험:
- 실제 리서치에서 추가해야 할 데이터:
""")


최종 체인 구성 후 호출

In [59]:
briefing_chain = briefing_prompt | llm | StrOutputParser()

result = briefing_chain.invoke({
    "student_keyword_text": student_keyword_text,
    "expanded_keyword_text": expanded_keyword_text,
    "selected_rss_text": selected_rss_text,
    "routing_summary": routing_summary,
    "news_text": news_text
})

최종결과 출력

In [ ]:
print("\n" + "=" * 100)
print("AI 최종 브리핑 결과")
print("=" * 100)
print(result)